# YOLO26 단안 깊이 추정 실습

한 장의 RGB 이미지에서 미터 단위 깊이 맵을 만들고, 값을 분석하고, 두 방식으로 시각화합니다. 공식 [Ultralytics Depth 문서](https://docs.ultralytics.com/tasks/depth/)를 바탕으로 구성했습니다.

## 0. 설치

아래 셀은 현재 커널에 패키지가 없을 때만 주석을 해제해 실행하세요. 설치 후 커널을 다시 시작해야 할 수 있습니다.

In [ ]:
# %pip install -U 'ultralytics>=8.4.115' opencv-python matplotlib

## 1. 환경과 장치 확인

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.utils.plotting import colorize_depth

print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('사용 장치:', DEVICE)

## 2. 사전학습 모델로 추론

첫 실행에는 `yolo26n-depth.pt`가 자동 다운로드됩니다. 공식 가중치의 학습 해상도인 768을 사용합니다.

In [ ]:
MODEL_NAME = 'yolo26n-depth.pt'
SOURCE = 'https://ultralytics.com/images/bus.jpg'

model = YOLO(MODEL_NAME)
result = model.predict(SOURCE, imgsz=768, device=DEVICE)[0]
depth = result.depth.data.cpu().numpy().astype(np.float32)
print('shape:', depth.shape, 'dtype:', depth.dtype)

## 3. 수치 분석

깊이는 미터 단위입니다. 0 이하, NaN, 무한대는 유효하지 않은 값으로 제외합니다.

In [ ]:
valid = np.isfinite(depth) & (depth > 0)
values = depth[valid]
print(f'유효 픽셀: {valid.mean():.2%}')
print('min / p10 / median / p90 / max (m):')
print(np.percentile(values, [0, 10, 50, 90, 100]))

h, w = depth.shape
print(f'중심 픽셀 ({w//2}, {h//2}): {depth[h//2, w//2]:.3f} m')

## 4. 상대 깊이와 절대 깊이 시각화

`disparity`는 역깊이로 가까운 곳을 따뜻하게 표시합니다. `metric`은 0~20 m를 선형으로 나타내므로 먼 곳이 따뜻합니다. OpenCV 결과는 BGR이므로 Matplotlib 표시 전에 RGB로 바꿉니다.

In [ ]:
relative_bgr = colorize_depth(depth, cmap='spectral', mode='disparity')
metric_bgr = colorize_depth(depth, vmin=0, vmax=20, cmap='inferno', mode='metric')
overlay_bgr = result.plot()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, image, title in zip(
    axes,
    [relative_bgr, metric_bgr, overlay_bgr],
    ['상대 깊이(disparity)', '절대 깊이 0~20 m(metric)', '원본 오버레이'],
):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()

## 5. 결과 저장

In [ ]:
from pathlib import Path

output_dir = Path('../outputs/notebook')
output_dir.mkdir(parents=True, exist_ok=True)
np.save(output_dir / 'depth_raw.npy', depth)
cv2.imwrite(str(output_dir / 'depth_colored.png'), relative_bgr)
cv2.imwrite(str(output_dir / 'depth_metric.png'), metric_bgr)
result.save(filename=str(output_dir / 'depth_overlay.png'))
print(output_dir.resolve())

## 6. 간단한 장애물 후보 마스크

3 m보다 가까운 유효 픽셀을 표시합니다. 이것은 개념 실습일 뿐 안전 시스템의 충돌 판단기로 사용하면 안 됩니다.

In [ ]:
threshold_m = 3.0
near_mask = valid & (depth < threshold_m)
print(f'{threshold_m} m 이내 픽셀 비율: {near_mask.mean():.2%}')
plt.figure(figsize=(10, 6))
plt.imshow(near_mask, cmap='gray')
plt.title(f'{threshold_m} m 이내 후보 영역')
plt.axis('off');

## 연습 문제

1. `SOURCE`를 자신의 이미지 경로로 바꾸고 결과를 비교하세요.
2. `threshold_m`를 2, 5, 10으로 바꿔 마스크 면적 변화를 표로 만드세요.
3. 한 픽셀이 아니라 20×20 관심 영역의 중앙 깊이를 계산하세요.
4. `yolo26s-depth.pt`로 바꿔 처리 시간과 시각적 품질을 비교하세요.
5. 반사체, 유리, 하늘, 텍스처가 없는 벽에서 실패 사례를 기록하세요.